# 04 | Proposed Randomized Experiment

This is a design proposal, not an executed experiment. Run notebook 01 first to generate the local risk table. The original design uses in-sample scores and the listed plan price as a value proxy, so its historical risk and pool-size estimates require prospective validation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Run from the repository root or notebooks/ directory.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
PARQ = DATA / "parquet"
OUTPUT = ROOT / "outputs"
OUTPUT.mkdir(exist_ok=True)

from scipy.stats import norm
d = pd.read_parquet(OUTPUT / 'experiment_risk_input.parquet')

## Risk-based eligibility and sample size

The original filter combines the top 30% of predicted risk, recent listening (within 14 days), and listed price of at least TWD 149. Eligibility does not imply responsiveness. The two-proportion normal approximation assumes equal group sizes, a two-sided 5% significance level and 80% power, without multiple-comparison adjustment.

In [ ]:
thr=d['risk'].quantile(0.70)
tgt=d[(d['risk']>=thr)&(d['last_ul_days_s']<=14)&(d['last_price']>=149)]; p0=tgt['is_churn'].mean()
def sample_per_arm(p0,de,alpha=0.05,power=0.8):
    p1=p0-de; pb=(p0+p1)/2
    z=norm.ppf(1-alpha/2)*np.sqrt(2*pb*(1-pb))+norm.ppf(power)*np.sqrt(p0*(1-p0)+p1*(1-p1)); return int(np.ceil((z/de)**2))
print(f'Candidate cohort: {len(tgt):,} users; observed churn: {p0:.3f}(overall {d["is_churn"].mean():.3f} ; {p0/d["is_churn"].mean():.1f} x)')
for de in [0.02,0.03,0.04]: print(f'  Detect {int(de*100)}pp reduction: per arm {sample_per_arm(p0,de):,} users')

## Historical planning estimates (recorded, not rerun)

Candidate pool: 167,423; observed churn: 29.1% (full-pipeline version). A separate decision notebook using stored models reported 149,894 candidates and 31.3% observed churn; those estimates are not combined.

| Detectable absolute reduction | Approximate users per arm |
|---|---:|
| 2 percentage points | 7,923 |
| 3 percentage points | 3,482 |
| 4 percentage points | 1,936 |

## Proposed design

Randomize eligible users within prespecified strata (risk, recent engagement, or payment stability) to no subsidy, discount, or free trial. Predefine the primary comparisons, allocation, follow-up period and minimum detectable effects; recalculate power with a multiplicity plan. Measure churn/renewal and longer-term incremental customer value and ROI, with revenue cannibalization and post-trial churn as guardrails. Use experimental data for future uplift estimation only after collecting and validating it.